In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from scipy.signal import hilbert, spectrogram
import glob
from scipy.stats import sem  # Standard error of the mean
from scipy.stats import sem


1) Create a Jupyter notebook that does the following:
    Loads a .wav file into python.
    Compute and plot a spectrogram for the .wav file.
    Compute and plot the broadband amplitude envelope using instructions below.
    Save the broadband amplitude envelope using the extension *_env.txt.


In [ ]:
root_dir = '/content/drive/My Drive/'
import os
print(os.listdir(root_dir))
data_dir = os.path.join(root_dir, 'Data', 'Stimuli', 'SR2_N.wav')

In [ ]:

# 1: Load the .wav file
wav_file_path = 'SR2_N.wav'  # Replace with your actual filename
sample_rate, audio_data = wavfile.read(wav_file_path)

# Convert to mono if stereo
if audio_data.ndim > 1:
    audio_data = np.mean(audio_data, axis=1)

# 2: Compute and plot the spectrogram
f, t, Sxx = spectrogram(audio_data, fs=sample_rate, nperseg=1024, noverlap=512)

plt.figure(figsize=(10, 4))
plt.pcolormesh(t, f, 10 * np.log10(Sxx + 1e-10), shading='gouraud')
plt.colorbar(label='Power (dB)')
plt.title('Spectrogram')
plt.ylabel('Frequency [Hz]')
plt.xlabel('Time [s]')
plt.tight_layout()
plt.show()

# 3: Compute the broadband amplitude envelope
analytic_signal = Sxx.mean()#hilbert(audio_data)
amplitude_envelope = np.abs(analytic_signal)

# Plot the envelope
time_vector = np.arange(len(amplitude_envelope)) / sample_rate
plt.figure(figsize=(10, 3))
plt.plot(time_vector, amplitude_envelope)
plt.title('Broadband Amplitude Envelope')
plt.xlabel('Time [s]')
plt.ylabel('Amplitude')
plt.tight_layout()
plt.show()

# 4: Save the envelope to a .txt file
envelope_filename = wav_file_path.replace('.wav', '_env.txt')
np.savetxt(envelope_filename, amplitude_envelope)

print(f"Envelope saved to: {envelope_filename}")



Color (in the Spectrogram)
Represents: The power or intensity of a frequency at a specific time.

Bright colors (yellow/green):
→ Strong signal (high energy or louder sound at that frequency and time).

Dark colors (blue/purple):
→ Weak signal (low energy or quieter sound).

Frequency
Represents: How high or low a sound is.

Low frequency = Deep or bassy sounds (e.g., drum beat, hum).

High frequency = Sharp or squeaky sounds (e.g., whistle, "s" in speech).

In the Spectrogram: Amplitude Becomes Power (Color)
The color intensity in the spectrogram is a visual representation of how much amplitude (energy) exists at each frequency and time.

Bright colors = more amplitude at that frequency and time.

Dark colors = less amplitude.

So, more amplitude → brighter color in that time-frequency point.

The top plot is like a heat map of the sound. It shows how different tones (from low to high pitch) change over time. Bright areas mean the sound is strong at that pitch and moment, while dark areas mean it’s quieter. For example, around 0.2 seconds and 1.2 seconds, there are bright yellow spots — those are probably where someone is speaking or making a clear sound.

The bottom plot shows how loud the sound is over time. Each peak means something was said or a sound was made, and flat areas mean there was silence or a pause. This plot gives us a simple way to see when sounds happen, kind of like tracking the rhythm or beats in music. T

2) Create a Jupyter notebook that does the following:
    Load all amplitude envelopes.
    Extract an “epoch” around the onset of the critical period from each amplitude envelope (same  length as the EEG epochs).
    Compute and plot the average amp. envelope and a confidence interval for the amp. envelope.


In [ ]:
# Parameters
sample_rate = 22050  # Hz
epoch_start = -0.2   # seconds
epoch_end = 0.8      # seconds
epoch_length = int((epoch_end - epoch_start) * sample_rate)

# Load envelope
envelope = np.loadtxt("SR2_N_env.txt")
onset_time = 0.5  # seconds (adjust if known more precisely)
onset_sample = int(onset_time * sample_rate)

# Compute window
start = onset_sample + int(epoch_start * sample_rate)
end = onset_sample + int(epoch_end * sample_rate)

# Handle boundary condition
if start >= 0 and end <= len(envelope):
    epoch = envelope[start:end]
    epochs = np.array([epoch])

    # Compute average and 95% CI
    mean_env = np.mean(epochs, axis=0)
    ci = sem(epochs, axis=0) * 1.96

    # Time axis
    time_axis = np.linspace(epoch_start, epoch_end, epoch_length)

    # Plot
    plt.figure(figsize=(10, 4))
    plt.plot(time_axis, mean_env, label='Amplitude Envelope')
    plt.fill_between(time_axis, mean_env - ci, mean_env + ci, alpha=0.3, label='95% CI')
    plt.axvline(0, color='gray', linestyle='--', label='Onset')
    plt.title("Amplitude Envelope Epoch")
    plt.xlabel("Time (s)")
    plt.ylabel("Amplitude")
    plt.legend()
    plt.tight_layout()
    plt.show()

else:
    print(f" Epoch window [{start}:{end}] is out of bounds. Envelope length: {len(envelope)}")


Note For ENV.text:

Each value tells you how loud the audio is at a specific moment:

Small values ≈ silence or quiet regions

Large values ≈ strong audio activity (e.g., spoken word)

Evelope - numerical representation of how the loudness of an audio signal changes over time